# 10 — Replay → HybridPpoStep → masked PPO boundary

Эта тетрадка берёт replay из notebook 09, превращает его в `TextTrajectoryBatch`, затем сразу поднимает до нового `HybridPpoStep`. Цель — проверить contract, masks и PPO actor-loss boundary без запуска LLM или critic runtime.


In [ ]:
from pathlib import Path
import json

import jax.numpy as jnp

from tunix_craftext.hybrid_rollout import (
    compute_masked_step_token_ppo_loss,
    hybrid_step_from_text_trajectory,
    hybrid_trajectory_from_steps,
)
from tunix_craftext.replay import ReplayArtifact, ReplayStep
from tunix_craftext.text_trajectory import text_trajectory_from_replay


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run inside tunix-craftext.')

path = ROOT / 'artifacts/trajectories/batched-qwen-rollout/env-0.json'
if not path.is_file():
    raise FileNotFoundError('Run notebook 09 first: ' + str(path))

payload = json.loads(path.read_text())
steps = tuple(ReplayStep(**step) for step in payload['steps'])
replay = ReplayArtifact(payload['config_path'], payload['commit'], payload['backend'], steps, payload['schema'])
batch = text_trajectory_from_replay(replay)

print('token ids:', batch.token_ids.shape)
print('prompt ids:', batch.prompt_token_ids.shape)
print('token mask:', batch.token_mask.tolist())
print('policy mask:', batch.policy_mask.tolist(), '(fallback rows stay in evidence but not actor loss)')


In [ ]:
# Replay-only example: the critic slot is explicit and shaped exactly as production expects.
# In notebook 12 this tensor comes from Gemma/Tunix separate critic scoring.
values = jnp.zeros((batch.token_ids.shape[0],), dtype=jnp.float32)
hybrid_step = hybrid_step_from_text_trajectory(batch, values=values)
hybrid_trajectory = hybrid_trajectory_from_steps([hybrid_step])

print('HybridPpoStep action ids:', hybrid_step.action_ids.tolist())
print('generation token mask:', hybrid_step.generation_token_mask.tolist())
print('actor-loss token mask:', hybrid_step.actor_loss_token_mask.tolist())
print('step masks [T, B]:', hybrid_trajectory.step_masks.tolist())


In [ ]:
step_rewards = jnp.sum(batch.rewards, axis=-1)
advantages = step_rewards - hybrid_step.values
loss = compute_masked_step_token_ppo_loss(
    actor_log_probs=hybrid_step.actor_log_probs,
    old_log_probs=batch.old_logprobs,
    generation_token_mask=hybrid_step.actor_loss_token_mask,
    advantages=advantages,
    step_valid_mask=hybrid_step.step_mask,
    clip_epsilon=0.2,
)

print('step rewards:', step_rewards.tolist())
print('advantages:', advantages.tolist())
print('hybrid masked PPO actor loss:', float(loss))
assert bool(jnp.isfinite(loss))


Итог: replay теперь не прыгает напрямую в research learner. Между replay evidence и PPO objective есть явный `HybridPpoStep`: token logprobs, values, generated-token mask, actor-learning mask и step mask проверены до learner/update слоя.
